# Catching a fixed sklearn bug from its formulas

sklearn 1.9 fixed a bug in `BayesianRidge`: the predicted uncertainty
(`return_std=True`) forgot to center the test features, so the
variance came out wrong away from the training mean
([#33918](https://github.com/scikit-learn/scikit-learn/pull/33918)).
The fix is a few lines deep inside `predict`. Instead of reading the
diff, we trace both versions and let the formulas show the change.

### Let's start with `sklearn == 1.8.0`. We will be analysing it's `BayesianRidge` function!

In [1]:
from skverify import to_sympy

from sklearn.linear_model import BayesianRidge
import numpy as np

In [2]:
rng = np.random.default_rng(100)

X = np.random.random((6, 6))
y = np.squeeze(2 * X[:1] + rng.normal())

Small random data, and the model is fitted with plain sklearn
outside the trace: the bug lives in `predict`, so that is the only
part we trace.

In [3]:
X.shape, y.shape

((6, 6), (6,))

In [4]:
model = BayesianRidge().fit(X, y)

Trace `predict` and look at the standard deviation it returns.
The textbook form is $\sqrt{x^\top \Sigma\, x + \sigma^2}$ with
$x$ centered by the training means. Watch what 1.8 actually computes:

In [21]:
def predict(X, y):
    _, stdev = model.predict(X, return_std=True)
    return stdev

out = to_sympy(predict, X, y)
out.formula

sqrt(Sum(X[i, j]*Sum(X[i, k]*const_0[k, j], (k, 0, 5)), (j, 0, 5)) + 0.19709153691281)

In [26]:
out.unchecked

(('const_0',
  (('table', 'concrete'),),
  ('const_0[i, j]',
   'const_0 = [[0.0009902957409994718, -2.3891758288486333e-07, -1.0236941245337835e-06, -6.964857598235732e-07, 6.731008620574767e-07, -2.4702189566545766e-07], [-2.389175828848338e-07, 0.000990301412995446, 5.296126973727268e-07, 7.078368838681406e-07, 3.674210499399606e-07, -1.1119747721669172e-06], [-1.0236941245337082e-06, 5.296126973727417e-07, 0.0009890704841343727, -1.0205761214198337e-06, 7.02788233876092e-07, 1.2416535522510817e-06], [-6.964857598235378e-07, 7.078368838681399e-07, -1.0205761214198763e-06, 0.000990332196062194, 1.7984296033722837e-07, 6.055057474987029e-07], [6.731008620574464e-07, 3.674210499399333e-07, 7.027882338761136e-07, 1.7984296033722435e-07, 0.0009887647915559571, 4.2035800399568744e-07], [-2.4702189566546613e-07, -1.1119747721669196e-06, 1.241653552251105e-06, 6.055057474987151e-07, 4.203580039957293e-07, 0.0009892031585899943]]')),)

`const0` is the fitted posterior covariance $\Sigma$ (its values
are disclosed in `out.unchecked`), and the constant under the root is
the noise variance. Now look closer: the quadratic form is in the raw
$X_{i,j}$. Nothing is centered anywhere. That is the bug: the training
features were centered during `fit`, so the test features must be
centered the same way in `predict`, and in 1.8 they were not.

Now the same trace on sklearn `1.9.0`:

In [5]:
import sklearn
sklearn.__version__

'1.9.0'

In [6]:
def predict(X, y):
    _, stdev = model.predict(X, return_std=True)
    return stdev

out = to_sympy(predict, X, y)
out.formula

sqrt(Sum((X[i, j] - const_0[j])*Sum((X[i, k] - const_0[k])*const_1[k, j], (k, 0, 5)), (j, 0, 5)) + 0.0456335792036574)

In [7]:
out.unchecked

(('const_0',
  (('table', 'concrete'),),
  ('const_0[i]',
   'const_0 = [0.6487705674113619, 0.3284410805297529, 0.3835076200710605, 0.20075344607391418, 0.4186130886276873, 0.42471613610347947]')),
 ('const_1',
  (('table', 'concrete'),),
  ('const_1[i, j]',
   'const_1 = [[0.09506816209152737, -0.026384902466023843, 0.005338593010213623, 0.042592583202141036, 0.011687591169225834, -0.03713610887392956], [-0.026384902466023843, 0.1278043327151104, 0.013312408546928924, -0.014168598483095625, -0.03983647690593472, 0.0232099634406986], [0.005338593010213624, 0.013312408546928927, 0.13611625668293884, -0.025048825135073408, 0.039725519352438905, -0.013520731912150242], [0.042592583202141036, -0.014168598483095629, -0.025048825135073415, 0.13839682756484536, 0.0026990272060496232, 0.008965647963007741], [0.011687591169225835, -0.039836476905934726, 0.0397255193524389, 0.002699027206049627, 0.09110391819425238, 0.002612445684104083], [-0.03713610887392957, 0.0232099634406986, -0.0135207319

The centering appeared: every factor is now
$(X_{i,j} - \mathtt{const0}_j)$ with `const0` the training feature
means, which is exactly what the fix added. The two formulas state the
bug and the fix more plainly than the diff does: in 1.8 the variance
is only correct at the training mean, and the further your test point
sits from it, the more the old formula overstates or understates the
uncertainty.

Had the bug not been fixed yet, this same pair of traces would have
found it: one look at the first formula shows the missing centering,
especially to someone who knows the textbook form.

Here is the link to the actual bug report: https://github.com/scikit-learn/scikit-learn/pull/33918